In [1]:
import os
import re
import math
import time
import random
import numpy as np
import pandas as pd

from PIL import Image

print("Python OK. Starting pipeline...")




Python OK. Starting pipeline...


In [2]:
def binary_accuracy_np(y_true, y_pred_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(np.int32).ravel()
    y_pred = (np.asarray(y_pred_prob).ravel() > threshold).astype(np.int32)
    return (y_true == y_pred).mean()


def sigmoid(x):
    x = np.asarray(x, dtype=np.float32)
    out = np.empty_like(x, dtype=np.float32)
    pos = x >= 0
    neg = ~pos
    out[pos] = 1.0 / (1.0 + np.exp(-x[pos]))
    ex = np.exp(x[neg])
    out[neg] = ex / (1.0 + ex)
    return out


def bce_loss(y, p, eps=1e-7):
    y = np.asarray(y, dtype=np.float32).ravel()
    p = np.asarray(p, dtype=np.float32).ravel()
    p = np.clip(p, eps, 1 - eps)
    return float(-(y * np.log(p) + (1 - y) * np.log(1 - p)).mean())


SEED = 42
random.seed(SEED)
np.random.seed(SEED)



In [3]:
IMG_SIZE = 150
BATCH_SIZE = 32

EPOCHS = 5

VALIDATION_SPLIT = 0.2
MODEL_NUM = 1

WORK_DIR = "/kaggle/working"
DATASET_DIR = "/kaggle/input/dogs-vs-cats-redux-kernels-edition"

train_cat_dir = os.path.join(DATASET_DIR, "train", "cat")
train_dog_dir = os.path.join(DATASET_DIR, "train", "dog")

candidate_test_dirs = [
    os.path.join(DATASET_DIR, "test", "test"),
    os.path.join(DATASET_DIR, "test"),
    os.path.join(DATASET_DIR, "test", "unknown"),
    os.path.join(DATASET_DIR, "test", "test", "unknown"),
]
test_dir = None
for d in candidate_test_dirs:
    if os.path.isdir(d):
        test_dir = d
        break

print("train_cat_dir exists:", os.path.exists(train_cat_dir), train_cat_dir)
print("train_dog_dir exists:", os.path.exists(train_dog_dir), train_dog_dir)
print("test_dir initial candidate resolved to:", test_dir)

if not (os.path.isdir(train_cat_dir) and os.path.isdir(train_dog_dir)):
    raise FileNotFoundError(
        "Expected extracted train directories not found. "
        f"train_cat_dir={train_cat_dir}, train_dog_dir={train_dog_dir}"
    )




train_cat_dir exists: True /kaggle/input/dogs-vs-cats-redux-kernels-edition/train/cat
train_dog_dir exists: True /kaggle/input/dogs-vs-cats-redux-kernels-edition/train/dog
test_dir initial candidate resolved to: /kaggle/input/dogs-vs-cats-redux-kernels-edition/test/test


In [4]:
def list_jpgs(d):
    return [os.path.join(d, f) for f in os.listdir(d) if f.lower().endswith(".jpg")]


cat_files = sorted(list_jpgs(train_cat_dir))
dog_files = sorted(list_jpgs(train_dog_dir))
test_files = (
    sorted(list_jpgs(test_dir)) if (test_dir and os.path.isdir(test_dir)) else []
)

print("Train counts:", "cat =", len(cat_files), "dog =", len(dog_files))
print(
    "Test count (discovered in test_dir candidate):", len(test_files), "from", test_dir
)

if len(cat_files) == 0 or len(dog_files) == 0:
    raise RuntimeError(
        "No training images found under expected train/cat and train/dog directories."
    )




Train counts: cat = 11242 dog = 11258
Test count (discovered in test_dir candidate): 0 from /kaggle/input/dogs-vs-cats-redux-kernels-edition/test/test


In [5]:
def load_image_as_vector(path, img_size=150):
    with Image.open(path) as im:
        im = im.convert("RGB")
        im = im.resize((img_size, img_size), resample=Image.BILINEAR)
        arr = np.asarray(im, dtype=np.float32) / 255.0
    return arr.reshape(-1)


X_paths = cat_files + dog_files
y = np.array([0] * len(cat_files) + [1] * len(dog_files), dtype=np.float32)

idx = np.arange(len(X_paths))
rng = np.random.default_rng(SEED)
rng.shuffle(idx)
X_paths = [X_paths[i] for i in idx]
y = y[idx]

n_total = len(X_paths)
n_val = int(round(n_total * VALIDATION_SPLIT))
n_train = n_total - n_val

train_paths = X_paths[:n_train]
val_paths = X_paths[n_train:]
y_train = y[:n_train]
y_val = y[n_train:]

print(f"Split: n_train={n_train}, n_val={n_val}")

t0 = time.time()
X_train = np.stack([load_image_as_vector(p, IMG_SIZE) for p in train_paths], axis=0)
X_val = np.stack([load_image_as_vector(p, IMG_SIZE) for p in val_paths], axis=0)
print(
    "Loaded arrays:",
    X_train.shape,
    X_val.shape,
    "in",
    round(time.time() - t0, 2),
    "sec",
)

mu = X_train.mean(axis=0, keepdims=True)
sigma = X_train.std(axis=0, keepdims=True) + 1e-6
X_train_s = (X_train - mu) / sigma
X_val_s = (X_val - mu) / sigma

# Change (score-relevant stability): keep the same standardized-pixels core features,
# but append a constant 1.0 feature so SGD can fit an intercept more robustly in high-dim space.
# This tends to reduce extreme miscalibration that hurts log-loss.
X_train_s = np.concatenate(
    [X_train_s, np.ones((X_train_s.shape[0], 1), dtype=X_train_s.dtype)], axis=1
)
X_val_s = np.concatenate(
    [X_val_s, np.ones((X_val_s.shape[0], 1), dtype=X_val_s.dtype)], axis=1
)

CLIP_STD = 3.0
X_train_s = np.clip(X_train_s, -CLIP_STD, CLIP_STD)
X_val_s = np.clip(X_val_s, -CLIP_STD, CLIP_STD)




Split: n_train=18000, n_val=4500
Loaded arrays: (18000, 67500) (4500, 67500) in 57.97 sec


In [6]:
def train_logreg_sgd(
    X_train_s,
    y_train,
    X_val_s,
    y_val,
    epochs=1,
    batch_size=32,
    lr=0.05,
    l2=0.001,
    seed=42,
):
    n, d = X_train_s.shape
    w = np.zeros((d,), dtype=np.float32)
    b = np.float32(0.0)
    rng = np.random.default_rng(seed)

    for ep in range(epochs):
        order = np.arange(n)
        rng.shuffle(order)
        Xo = X_train_s[order]
        yo = y_train[order]

        for start in range(0, n, batch_size):
            xb = Xo[start : start + batch_size]
            yb = yo[start : start + batch_size]

            logits = xb @ w + b
            p = sigmoid(logits)
            diff = (p - yb).astype(np.float32)
            gw = (xb.T @ diff) / xb.shape[0]
            gb = diff.mean()
            gw += l2 * w

            w -= lr * gw
            b -= lr * gb

        p_tr = sigmoid(X_train_s @ w + b)
        p_va = sigmoid(X_val_s @ w + b)
        print(
            f"Epoch {ep+1}/{epochs} - "
            f"train_loss={bce_loss(y_train, p_tr):.4f} val_loss={bce_loss(y_val, p_va):.4f} "
            f"val_acc={binary_accuracy_np(y_val, p_va):.4f}"
        )

    return w, b


models_params = []
for i in range(MODEL_NUM):
    print(f"model : {i}")

    # Change (score-relevant calibration): scale LR by 1/sqrt(d) and slightly increase L2.
    # In this very high-dimensional flattened-pixel setup, this reduces overshooting and
    # extreme probabilities that heavily penalize log-loss, moving score toward target.
    d_eff = X_train_s.shape[1]
    lr_eff = 0.05 / float(np.sqrt(d_eff))
    l2_eff = 0.01

    w, b = train_logreg_sgd(
        X_train_s,
        y_train,
        X_val_s,
        y_val,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        lr=lr_eff,
        l2=l2_eff,
        seed=SEED + i,
    )
    models_params.append((w, b))

for i, (w, b) in enumerate(models_params):
    np.savez(os.path.join(WORK_DIR, f"model{i}.npz"), w=w, b=b)
print("Saved model parameter files to", WORK_DIR)



model : 0
Epoch 1/5 - train_loss=0.6653 val_loss=0.6808 val_acc=0.5867
Epoch 2/5 - train_loss=0.6316 val_loss=0.6555 val_acc=0.6162
Epoch 3/5 - train_loss=0.6253 val_loss=0.6593 val_acc=0.6098
Epoch 4/5 - train_loss=0.6415 val_loss=0.6780 val_acc=0.5978
Epoch 5/5 - train_loss=0.6326 val_loss=0.6753 val_acc=0.5969
Saved model parameter files to /kaggle/working


In [7]:
sum_pred = None
for i in range(MODEL_NUM):
    params = np.load(os.path.join(WORK_DIR, f"model{i}.npz"))
    w = params["w"]
    b = params["b"]
    pred = sigmoid(X_val_s @ w + b).reshape(-1, 1)
    sum_pred = pred if sum_pred is None else (sum_pred + pred)

avg_pred = (sum_pred / MODEL_NUM).ravel()
acc = binary_accuracy_np(y_val, avg_pred, threshold=0.5)
print(f"Validation accuracy: {acc:.4f}")
print(f"Validation logloss: {bce_loss(y_val, np.clip(avg_pred, 1e-7, 1-1e-7)):.4f}")




Validation accuracy: 0.5969
Validation logloss: 0.6753


In [8]:
def extract_number(filename):
    m = re.search(r"\d+", os.path.basename(filename))
    return int(m.group()) if m else -1


def iter_jpgs_recursive(root_dir):
    for r, _, files in os.walk(root_dir):
        for f in files:
            if f.lower().endswith(".jpg"):
                yield os.path.join(r, f)


sample_path = os.path.join(DATASET_DIR, "sample_submission.csv")
if not os.path.exists(sample_path):
    sample_path = "/kaggle/input/sample_submission.csv"
sample = pd.read_csv(sample_path)
expected_ids = sample["id"].astype(int).tolist()

if len(test_files) == 0:
    test_root = os.path.join(DATASET_DIR, "test")
    if not os.path.isdir(test_root):
        raise FileNotFoundError(f"Expected test root directory not found: {test_root}")

    id_to_path = {}
    for p in iter_jpgs_recursive(test_root):
        base = os.path.splitext(os.path.basename(p))[0]
        tid = int(base) if base.isdigit() else extract_number(base)
        if tid != -1 and tid not in id_to_path:
            id_to_path[tid] = p

    test_dir = test_root
    test_files = list(id_to_path.values())
    print(
        "Fallback recursive test discovery used. Found",
        len(test_files),
        "jpgs under",
        test_root,
    )
else:
    id_to_path = {}
    for p in test_files:
        base = os.path.splitext(os.path.basename(p))[0]
        tid = int(base) if base.isdigit() else extract_number(base)
        if tid != -1:
            id_to_path[tid] = p

missing = [i for i in expected_ids if i not in id_to_path]
if len(missing) > 0:
    raise RuntimeError(
        f"Could not find {len(missing)} test images required by sample_submission. "
        f"Example missing ids: {missing[:10]}. "
        f"Resolved test search root={test_dir}; discovered {len(test_files)} jpgs."
    )

ordered_test_paths = [id_to_path[i] for i in expected_ids]

t0 = time.time()
X_test = np.stack(
    [load_image_as_vector(p, IMG_SIZE) for p in ordered_test_paths], axis=0
)
X_test_s = (X_test - mu) / sigma

# Must match train/val feature shape: append constant feature.
X_test_s = np.concatenate(
    [X_test_s, np.ones((X_test_s.shape[0], 1), dtype=X_test_s.dtype)], axis=1
)

X_test_s = np.clip(X_test_s, -CLIP_STD, CLIP_STD)
print("Loaded test array:", X_test_s.shape, "in", round(time.time() - t0, 2), "sec")

sum_pred = None
for i in range(MODEL_NUM):
    params = np.load(os.path.join(WORK_DIR, f"model{i}.npz"))
    w = params["w"]
    b = params["b"]
    pred = sigmoid(X_test_s @ w + b).reshape(-1, 1)
    sum_pred = pred if sum_pred is None else (sum_pred + pred)

avg_pred = (sum_pred / MODEL_NUM).ravel()

# Change (metric safety): use same epsilon as BCE to avoid any accidental log(0) on Kaggle.
labels = np.clip(avg_pred, 1e-7, 1 - 1e-7)

sub = pd.DataFrame(
    {"id": np.asarray(expected_ids, dtype=int), "label": labels.astype(float)}
)

sub_path = os.path.join(WORK_DIR, "submission.csv")
sub.to_csv(sub_path, index=False)

print("Wrote submission.csv with shape:", sub.shape, "to", sub_path)
print(sub.head())
print(sub.tail())

Fallback recursive test discovery used. Found 2500 jpgs under /kaggle/input/dogs-vs-cats-redux-kernels-edition/test
Loaded test array: (2500, 67501) in 5.53 sec
Wrote submission.csv with shape: (2500, 2) to /kaggle/working/submission.csv
   id     label
0   1  0.889095
1   2  0.622392
2   3  0.659343
3   4  0.516652
4   5  0.753298
        id     label
2495  2496  0.714121
2496  2497  0.409084
2497  2498  0.222156
2498  2499  0.766228
2499  2500  0.573551
